# Chapter 09 - Pipelines and ColumnTransformers

So far we have scaled numerical features and encoded categorical features as separate steps.
In practice, performing these steps manually is error-prone — it is easy to accidentally fit
a transformer on the test set, forget a step, or apply transformations in the wrong order.

Scikit-learn's `Pipeline` and `ColumnTransformer` solve this by bundling all preprocessing
and modeling steps into a single object that can be fit, transformed, and cross-validated
as a unit — with zero risk of data leakage.

**Topics covered:**
- The data leakage problem
- sklearn Pipeline: chaining steps
- ColumnTransformer: different transforms per column type
- Building a complete preprocessing pipeline
- make_pipeline() and make_column_transformer() shortcuts
- Pipelines with cross-validation
- Feature selection basics: SelectKBest, VarianceThreshold
- Full pipeline: raw data to prediction

## Setup

In [ ]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer
from sklearn.feature_selection import SelectKBest, f_regression, VarianceThreshold
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

np.random.seed(42)

## The Problem: Data Leakage from Preprocessing Before Splitting

Data leakage occurs when information from outside the training set is used to create
the model. The most common form in preprocessing:

```python
# WRONG: fitting the scaler on ALL data before splitting
X_scaled = StandardScaler().fit_transform(X)     # sees test data!
X_train, X_test = train_test_split(X_scaled)

# CORRECT: fit only on training data
X_train, X_test = train_test_split(X)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)          # fit on train
X_test = scaler.transform(X_test)                 # transform only
```

When you scale before splitting, the scaler's mean and standard deviation include
test data — subtly inflating your model's apparent performance.

In [ ]:
# Demonstration of the leakage difference
from sklearn.datasets import make_regression

X, y = make_regression(n_samples=200, n_features=5, noise=10, random_state=42)
X = pd.DataFrame(X, columns=[f'feature_{i}' for i in range(5)])

# WRONG approach: scale everything first
X_scaled_all = StandardScaler().fit_transform(X)
X_train_bad, X_test_bad, y_train, y_test = train_test_split(
    X_scaled_all, y, test_size=0.2, random_state=42
)

# CORRECT approach: split first, then scale
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
scaler = StandardScaler()
X_train_good = scaler.fit_transform(X_train_raw)
X_test_good = scaler.transform(X_test_raw)

# Compare: the leaky version has test mean closer to 0
print('Test set column means (should NOT be exactly 0):')
print(f'  Leaky approach:  {X_test_bad.mean(axis=0).round(3)}')
print(f'  Correct approach: {X_test_good.mean(axis=0).round(3)}')

## sklearn Pipeline: Chaining Preprocessing + Model

A `Pipeline` is an ordered sequence of transformers followed by an optional final
estimator (model). When you call `fit()` on the pipeline, it chains `.fit_transform()`
through each step, then calls `.fit()` on the final estimator.

This guarantees that each transformer is fit only on the data that flows through it
— no leakage, no forgotten steps.

In [ ]:
# A simple pipeline: impute missing values -> scale -> fit a model
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0)),
])

print('Pipeline steps:')
for name, step in pipe.steps:
    print(f'  {name}: {step.__class__.__name__}')

In [ ]:
# Inject some missing values to demonstrate imputation
X_train_missing = X_train_raw.copy()
mask = np.random.random(X_train_missing.shape) < 0.05
X_train_missing[mask] = np.nan

X_test_missing = X_test_raw.copy()
mask_test = np.random.random(X_test_missing.shape) < 0.05
X_test_missing[mask_test] = np.nan

print(f'Training NaNs: {X_train_missing.isna().sum().sum()}')
print(f'Test NaNs:     {X_test_missing.isna().sum().sum()}')

In [ ]:
# Fit the entire pipeline on training data
pipe.fit(X_train_missing, y_train)

# Predict on test data — imputation + scaling + prediction in one call
y_pred = pipe.predict(X_test_missing)

print(f'R2 score: {r2_score(y_test, y_pred):.3f}')
print(f'MAE:      {mean_absolute_error(y_test, y_pred):.2f}')

### Accessing Steps Inside a Pipeline

You can inspect individual steps by name or index.

In [ ]:
# Access by name
print('Imputer fill values:', pipe.named_steps['imputer'].statistics_.round(3))
print('Scaler means:       ', pipe.named_steps['scaler'].mean_.round(3))
print('Ridge coefficients: ', pipe.named_steps['model'].coef_.round(3))

# Access by index
print('\nFirst step:', pipe[0])

## ColumnTransformer: Different Transforms for Different Columns

Real datasets have a mix of numerical and categorical columns that require different
preprocessing. `ColumnTransformer` applies specified transformations to specified columns
and concatenates the results.

```
ColumnTransformer
├── numerical columns  -> SimpleImputer(mean) -> StandardScaler
├── categorical columns -> SimpleImputer(most_frequent) -> OneHotEncoder
└── [passthrough or drop remaining columns]
```

### Creating a Mixed Dataset

In [ ]:
np.random.seed(42)
n = 500

df = pd.DataFrame({
    'age': np.random.normal(40, 12, n).round(0),
    'income': np.random.lognormal(10.5, 0.8, n).round(0),
    'years_experience': np.random.uniform(0, 30, n).round(1),
    'education': np.random.choice(['high_school', 'bachelor', 'master', 'phd'], n,
                                  p=[0.30, 0.35, 0.25, 0.10]),
    'department': np.random.choice(['engineering', 'marketing', 'sales', 'hr', 'finance'], n),
    'is_manager': np.random.choice([True, False], n, p=[0.2, 0.8]),
})

# Target: performance score with signal from features
df['performance'] = (
    50
    + 0.3 * df['years_experience']
    + df['education'].map({'high_school': 0, 'bachelor': 5, 'master': 10, 'phd': 12})
    + df['is_manager'] * 8
    + np.random.normal(0, 5, n)
).round(1)

# Inject some missing values
df.loc[np.random.choice(n, 20, replace=False), 'income'] = np.nan
df.loc[np.random.choice(n, 15, replace=False), 'education'] = np.nan
df.loc[np.random.choice(n, 10, replace=False), 'years_experience'] = np.nan

print(f'Shape: {df.shape}')
print(f'\nMissing values:')
print(df.isna().sum().to_string())
print()
df.head()

In [ ]:
# Identify column types
numerical_cols = ['age', 'income', 'years_experience']
categorical_cols = ['education', 'department']
binary_cols = ['is_manager']

print(f'Numerical:   {numerical_cols}')
print(f'Categorical: {categorical_cols}')
print(f'Binary:      {binary_cols}')

In [ ]:
# Split first, preprocess after
X = df.drop(columns='performance')
y = df['performance']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'Train: {X_train.shape}, Test: {X_test.shape}')

### Building the ColumnTransformer

In [ ]:
# Define a sub-pipeline for numerical features
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

# Define a sub-pipeline for categorical features
categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

# Combine them in a ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_pipeline, numerical_cols),
        ('cat', categorical_pipeline, categorical_cols),
        ('bin', 'passthrough', binary_cols),  # binary columns need no transformation
    ],
    remainder='drop'  # drop any columns not listed above
)

print('ColumnTransformer structure:')
print(preprocessor)

In [ ]:
# Fit and transform the training data
X_train_processed = preprocessor.fit_transform(X_train)

print(f'Input shape:  {X_train.shape}')
print(f'Output shape: {X_train_processed.shape}')
print(f'\nOutput feature names:')
feature_names = preprocessor.get_feature_names_out()
print(list(feature_names))

In [ ]:
# Peek at the processed data
pd.DataFrame(
    X_train_processed[:5],
    columns=feature_names
).round(3)

### Full Pipeline: Preprocessor + Model

In [ ]:
# Combine the preprocessor with a model into one pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', Ridge(alpha=1.0)),
])

# Fit on training data
full_pipeline.fit(X_train, y_train)

# Predict on raw test data — all preprocessing happens automatically
y_pred = full_pipeline.predict(X_test)

print(f'R2 score: {r2_score(y_test, y_pred):.3f}')
print(f'MAE:      {mean_absolute_error(y_test, y_pred):.2f}')

## Shortcut Functions: make_pipeline and make_column_transformer

These convenience functions auto-generate step names, saving you from writing out
the `(name, transformer)` tuples.

In [ ]:
# make_pipeline auto-names steps based on class name
num_pipe = make_pipeline(SimpleImputer(strategy='median'), StandardScaler())
cat_pipe = make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder(handle_unknown='ignore', sparse_output=False))

print('Auto-generated step names:')
for name, step in num_pipe.steps:
    print(f'  {name}: {step.__class__.__name__}')

In [ ]:
# make_column_transformer auto-names transformers
preprocessor_short = make_column_transformer(
    (num_pipe, numerical_cols),
    (cat_pipe, categorical_cols),
    ('passthrough', binary_cols),
    remainder='drop'
)

# Combine with model using make_pipeline
short_pipeline = make_pipeline(preprocessor_short, Ridge(alpha=1.0))

short_pipeline.fit(X_train, y_train)
y_pred_short = short_pipeline.predict(X_test)

print(f'R2 score (shortcut version): {r2_score(y_test, y_pred_short):.3f}')
print('Same result — just less typing.')

## Using Pipelines with Cross-Validation (No Data Leakage)

The biggest advantage of pipelines: when combined with `cross_val_score`, all
preprocessing is performed **inside each fold**. The scaler is fit on the training
portion of each fold and applied to the validation portion — automatically.

In [ ]:
# Cross-validation with the full pipeline
cv_scores = cross_val_score(
    full_pipeline,
    X_train,
    y_train,
    cv=5,
    scoring='r2'
)

print(f'5-fold CV R2 scores: {cv_scores.round(3)}')
print(f'Mean R2: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

In [ ]:
# Compare with a different model — just swap the last step
rf_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(n_estimators=100, random_state=42)),
])

rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='r2')
print(f'Random Forest 5-fold CV R2: {rf_scores.mean():.3f} (+/- {rf_scores.std():.3f})')
print(f'Ridge         5-fold CV R2: {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

> Each cross-validation fold fits the preprocessor from scratch on the fold's training
> data. This means the scaler's mean/std and the encoder's categories are computed
> fresh for each fold — exactly as they would be in a real train/deploy scenario.

## Feature Selection Basics

Not all features contribute equally. Removing irrelevant or redundant features can
improve model performance, reduce overfitting, and speed up training.

### VarianceThreshold

Removes features with variance below a threshold. Features with near-zero variance
carry almost no information.

In [ ]:
# Create data with one near-constant feature
X_demo = pd.DataFrame({
    'useful_1': np.random.normal(0, 1, 100),
    'useful_2': np.random.normal(5, 2, 100),
    'near_constant': np.full(100, 3.0) + np.random.normal(0, 0.001, 100),
    'useful_3': np.random.uniform(0, 10, 100),
})

print('Variances before:')
print(X_demo.var().to_string())

# Remove features with variance below 0.01
selector = VarianceThreshold(threshold=0.01)
X_selected = selector.fit_transform(X_demo)

kept = selector.get_support()
print(f'\nKept columns: {X_demo.columns[kept].tolist()}')
print(f'Dropped:      {X_demo.columns[~kept].tolist()}')

### SelectKBest

Selects the top *k* features based on a statistical test (e.g., F-statistic for
regression, chi-squared for classification). This is a univariate method — it
evaluates each feature independently.

In [ ]:
# Use SelectKBest on the processed training data
X_train_proc = preprocessor.fit_transform(X_train)
feature_names_all = preprocessor.get_feature_names_out()

selector_k = SelectKBest(score_func=f_regression, k=5)
X_train_kbest = selector_k.fit_transform(X_train_proc, y_train)

# Show scores for all features
scores = pd.Series(selector_k.scores_, index=feature_names_all).sort_values(ascending=False)
print('Feature F-scores (higher = more relevant):')
print(scores.round(2).to_string())
print(f'\nSelected top 5: {list(feature_names_all[selector_k.get_support()])}')

### Feature Selection Inside a Pipeline

Feature selection should be treated like any other preprocessing step — fit on
training data only. Putting it in the pipeline guarantees this.

In [ ]:
pipeline_with_selection = Pipeline([
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_regression, k=5)),
    ('model', Ridge(alpha=1.0)),
])

cv_scores_sel = cross_val_score(
    pipeline_with_selection, X_train, y_train, cv=5, scoring='r2'
)

print(f'With feature selection:    R2 = {cv_scores_sel.mean():.3f} (+/- {cv_scores_sel.std():.3f})')
print(f'Without feature selection: R2 = {cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

## Putting It All Together: Raw Data to Prediction

Let us build a complete, production-ready pipeline that takes raw data with mixed types
and missing values, preprocesses it, selects features, and makes predictions.

In [ ]:
# Generate a larger, more realistic dataset
np.random.seed(42)
n = 1000

raw_data = pd.DataFrame({
    'square_feet': np.random.lognormal(7.2, 0.4, n).round(0),
    'lot_size': np.random.lognormal(8.5, 0.6, n).round(0),
    'year_built': np.random.randint(1950, 2024, n),
    'bedrooms': np.random.choice([1, 2, 3, 4, 5], n, p=[0.05, 0.20, 0.40, 0.25, 0.10]),
    'bathrooms': np.random.choice([1.0, 1.5, 2.0, 2.5, 3.0], n),
    'garage_spaces': np.random.choice([0, 1, 2, 3], n, p=[0.15, 0.30, 0.40, 0.15]),
    'neighborhood': np.random.choice(
        ['downtown', 'midtown', 'suburbs', 'countryside'], n,
        p=[0.25, 0.30, 0.30, 0.15]
    ),
    'condition': np.random.choice(['poor', 'fair', 'good', 'excellent'], n,
                                  p=[0.10, 0.25, 0.40, 0.25]),
    'has_pool': np.random.choice([True, False], n, p=[0.25, 0.75]),
    'has_fireplace': np.random.choice([True, False], n, p=[0.35, 0.65]),
})

# Target with realistic signal
raw_data['price'] = (
    50_000
    + raw_data['square_feet'] * 150
    + raw_data['lot_size'] * 5
    + (raw_data['year_built'] - 1950) * 500
    + raw_data['bedrooms'] * 10_000
    + raw_data['bathrooms'] * 15_000
    + raw_data['garage_spaces'] * 12_000
    + raw_data['neighborhood'].map({'downtown': 80_000, 'midtown': 40_000, 'suburbs': 10_000, 'countryside': -20_000})
    + raw_data['condition'].map({'poor': -40_000, 'fair': 0, 'good': 25_000, 'excellent': 50_000})
    + raw_data['has_pool'] * 30_000
    + raw_data['has_fireplace'] * 10_000
    + np.random.normal(0, 30_000, n)
).round(0)

# Inject missing values
for col in ['square_feet', 'lot_size', 'year_built', 'bathrooms']:
    raw_data.loc[np.random.choice(n, 30, replace=False), col] = np.nan
for col in ['neighborhood', 'condition']:
    raw_data.loc[np.random.choice(n, 20, replace=False), col] = np.nan

print(f'Shape: {raw_data.shape}')
print(f'\nMissing values:')
print(raw_data.isna().sum()[raw_data.isna().sum() > 0].to_string())
print()
raw_data.head()

In [ ]:
# Define column groups
num_features = ['square_feet', 'lot_size', 'year_built', 'bedrooms', 'bathrooms', 'garage_spaces']
cat_features = ['neighborhood']
ord_features = ['condition']
bin_features = ['has_pool', 'has_fireplace']

# Sub-pipelines
num_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

cat_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False, drop='first')),
])

ord_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(categories=[['poor', 'fair', 'good', 'excellent']])),
])

# Complete preprocessor
full_preprocessor = ColumnTransformer([
    ('num', num_transformer, num_features),
    ('cat', cat_transformer, cat_features),
    ('ord', ord_transformer, ord_features),
    ('bin', 'passthrough', bin_features),
])

print('Preprocessor defined.')
print(f'  Numerical ({len(num_features)}):   {num_features}')
print(f'  Categorical ({len(cat_features)}): {cat_features}')
print(f'  Ordinal ({len(ord_features)}):     {ord_features}')
print(f'  Binary ({len(bin_features)}):      {bin_features}')

In [ ]:
# Complete pipeline: preprocessor -> model
X = raw_data.drop(columns='price')
y = raw_data['price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

final_pipeline = Pipeline([
    ('preprocessor', full_preprocessor),
    ('model', Ridge(alpha=1.0)),
])

# Cross-validate on training data
cv_results = cross_val_score(final_pipeline, X_train, y_train, cv=5, scoring='r2')
print(f'5-fold CV R2: {cv_results.mean():.3f} (+/- {cv_results.std():.3f})')

In [ ]:
# Final fit on full training set, evaluate on test set
final_pipeline.fit(X_train, y_train)
y_pred = final_pipeline.predict(X_test)

print('Test set performance:')
print(f'  R2:  {r2_score(y_test, y_pred):.3f}')
print(f'  MAE: ${mean_absolute_error(y_test, y_pred):,.0f}')

# Show a sample of predictions
comparison = pd.DataFrame({'actual': y_test.values[:10], 'predicted': y_pred[:10].round(0)})
comparison['error'] = (comparison['predicted'] - comparison['actual']).round(0)
comparison

In [ ]:
# The beauty: prediction on brand new raw data is a single line
new_house = pd.DataFrame([{
    'square_feet': 1800,
    'lot_size': 6000,
    'year_built': 2005,
    'bedrooms': 3,
    'bathrooms': 2.0,
    'garage_spaces': 2,
    'neighborhood': 'suburbs',
    'condition': 'good',
    'has_pool': False,
    'has_fireplace': True,
}])

predicted_price = final_pipeline.predict(new_house)
print(f'Predicted price for new house: ${predicted_price[0]:,.0f}')

## Pipeline Best Practices

1. **Always use pipelines in production code.** Manual preprocessing is a guaranteed
   source of bugs and data leakage.

2. **Split before anything else.** The very first step in any ML workflow should be
   `train_test_split`. Everything after that should happen inside a pipeline.

3. **Use `cross_val_score` with the pipeline.** This ensures preprocessing is done
   independently in each fold.

4. **Set `handle_unknown='ignore'` on encoders.** Production data will eventually
   contain categories the model has never seen.

5. **Name your steps meaningfully.** When using `Pipeline()` directly (not `make_pipeline`),
   give steps descriptive names so that accessing `named_steps` is intuitive.

6. **Save the entire pipeline with joblib.** This bundles preprocessing and model into
   a single serialized object:
   ```python
   import joblib
   joblib.dump(final_pipeline, 'model_pipeline.pkl')
   loaded = joblib.load('model_pipeline.pkl')
   loaded.predict(new_data)
   ```

---

**Summary:** This notebook showed how to eliminate data leakage and manual preprocessing
errors by using scikit-learn Pipelines and ColumnTransformers. The key ideas:

- `Pipeline` chains transformers and a model into a single fit/predict object
- `ColumnTransformer` applies different transformations to different column types
- `make_pipeline` and `make_column_transformer` provide concise shortcuts
- Pipelines work seamlessly with `cross_val_score` for leakage-free evaluation
- Feature selection steps (SelectKBest, VarianceThreshold) belong inside the pipeline

With these tools, your entire workflow — from raw, messy data to final prediction — lives
in a single, reproducible, serializable object.